In [ ]:
import os
import shutil
from google.colab import drive, userdata

# Remove existing directory if it exists
if os.path.exists('/content/mini-gpt'):
    shutil.rmtree('/content/mini-gpt')

# Clone the repository from GitHub
!git clone https://github.com/maariogutierrez/mini-gpt.git
ROOT = '/content/mini-gpt'
os.chdir(ROOT)

# Mount Google Drive for data and outputs
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/My Drive/mini-gpt'

REQUIREMENTS = 'https://raw.githubusercontent.com/maariogutierrez/mini-gpt/main/requirements.txt'
!pip install -r $REQUIREMENTS

# wandb_api_key = userdata.get('WANDB')
# print("Logging into Weights & Biases (wandb). Follow the prompts.")
# import wandb
# wandb.login(key=wandb_api_key)

# Store outputs in Google Drive, but keep code in cloned repo
EXPORTS_DIR = os.path.join(DRIVE_ROOT, 'exports')
LOGS_DIR = os.path.join(DRIVE_ROOT, 'logs')
DATA_DIR = os.path.join(DRIVE_ROOT, 'data')
CHECKPOINTS_DIR = os.path.join(DRIVE_ROOT, 'checkpoints')

print("\n--- Setup Summary ---")
print(f"Code repository: {ROOT}")
print(f"Google Drive: {DRIVE_ROOT}")
print(f"Current Working Directory: {os.getcwd()}")

In [ ]:
import sys
import numpy as np
from pathlib import Path

# Add the ROOT to sys.path so we can import from model module
sys.path.insert(0, ROOT)

from model.architecture.tokenizer import CustomTokenizer
from model.training.preprocess import process_dataset

print("✓ Imports successful")

In [ ]:
print("Starting preprocessing pipeline...\n")
process_dataset(
    output_dir=DATA_DIR,
    train_split=0.9,
    dataset_name="roneneldan/TinyStories",
)
print("\n✓ Preprocessing complete!")

In [ ]:
print("\n" + "="*60)
print("FILE SIZE VERIFICATION")
print("="*60 + "\n")

train_bin_path = Path(DATA_DIR) / "train.bin"
val_bin_path = Path(DATA_DIR) / "val.bin"

if train_bin_path.exists():
    train_size_bytes = train_bin_path.stat().st_size
    train_size_mb = train_size_bytes / (1024**2)
    train_size_gb = train_size_bytes / (1024**3)
    train_tokens = train_size_bytes // 2  
    print(f"train.bin")
    print(f"  Size: {train_size_mb:.2f} MB ({train_size_gb:.4f} GB)")
    print(f"  Tokens: {train_tokens:,}")
else:
    print("❌ train.bin not found!")

if val_bin_path.exists():
    val_size_bytes = val_bin_path.stat().st_size
    val_size_mb = val_size_bytes / (1024**2)
    val_size_gb = val_size_bytes / (1024**3)
    val_tokens = val_size_bytes // 2  
    print(f"\nval.bin")
    print(f"  Size: {val_size_mb:.2f} MB ({val_size_gb:.4f} GB)")
    print(f"  Tokens: {val_tokens:,}")
else:
    print("❌ val.bin not found!")

if train_bin_path.exists() and val_bin_path.exists():
    total_tokens = train_tokens + val_tokens
    train_pct = train_tokens / total_tokens * 100
    print(f"\nTotal: {total_tokens:,} tokens")
    print(f"Split: {train_pct:.1f}% train, {100-train_pct:.1f}% validation")
    print("\n✓ Files verified successfully!")

In [ ]:
print("\n" + "="*60)
print("BATCH DECODING VERIFICATION")
print("="*60 + "\n")

val_tokens = np.memmap(val_bin_path, dtype=np.uint16, mode='r')

tokenizer = CustomTokenizer()

batch_size = 256
random_start = np.random.randint(0, len(val_tokens) - batch_size)
batch_tokens = val_tokens[random_start:random_start + batch_size]

decoded_text = tokenizer.decode(batch_tokens.tolist())

print(f"Random batch from validation set (tokens {random_start} to {random_start + batch_size}):\n")
print("-" * 60)
print(decoded_text)
print("-" * 60)
print(f"\n✓ Decoded text is readable! Batch successfully loaded and verified.")
print(f"  Input: {batch_size} tokens")
print(f"  Output length: {len(decoded_text)} characters")